# Breakout 1B · Sprint 2 · build your recipe's signal, then set its threshold

One recipe per group: **2** description, **3** kilos per unit, **4** generic description, **5** invoice. Run the setup cell, then the Segments section (it goes with the clustering chart), then only the section of your recipe, then the last section. Everything here uses **months 1 to 18**: months 19 to 36 are the test, after the lock. Stuck? `1B_compute_worked.xlsx` shows every formula.

*Published customs datasets record an irregularity on 1 to 8 lines in 100; in this lab the random draw finds 3 in 100.*

In [ ]:
# Run this cell first (click it, then Shift + Enter). It loads the lab data.
import hashlib, json, os
from pathlib import Path
import numpy as np, pandas as pd

# The folder that holds features_lines.csv. Change it here if your data is elsewhere.
CANDIDATES = [os.environ.get('WORKSHOP_DATA', ''), 'data', '../data', '/workspace/data', '.']
DATA = next(Path(c) for c in CANDIDATES if c and (Path(c) / 'features_lines.csv').exists())
lines = pd.read_csv(DATA / 'features_lines.csv', low_memory=False)
lines['found'] = (lines.outcome == 'finding').astype(int)
print('Data folder:', DATA, '·', f'{len(lines):,}', 'declaration lines')

def _h(v, nd):
    return hashlib.sha256(f'{round(float(v), nd):.{nd}f}'.encode()).hexdigest()[:16]
def check(name, value, expected, nd):
    ok = value is not None and value is not ... and _h(value, nd) == expected
    num = isinstance(value, (int, float, np.integer, np.floating))
    shown = (f'{value:.{nd}f}' if 'code' in name else f'{value:,.{nd}f}') if num else str(value)
    print(('✓ ' if ok else '✗ not yet: ') + name + ' = ' + shown)


In [ ]:
h1 = lines[lines.period_month <= 18]
tariff = pd.read_csv(DATA / 'tariff.csv')[['hs8', 'duty_mfn']]
h1 = h1.drop(columns=[c for c in ['duty_mfn'] if c in h1.columns]).merge(tariff, on='hs8', how='left')

## Segments · the importers in four groups, computed for you

With the clustering chart. k-means put the importers into four segments on six signals of months 1 to 18, 1 the quietest and 4 the riskiest (`importer_cluster`, given: k-means needs a script, not a spreadsheet). You compute what each segment finds on months 1 to 18, and which segment a valuation or documentary team looks at first.

**Hint 1:** `traders.groupby('importer_cluster')` gathers the rows of each segment; `.agg(...)` counts and adds; `.idxmax()` gives the segment with the highest value.

<details><summary>Hint 2</summary>

```python
by_segment = traders.groupby('importer_cluster').agg(importers=('importer_id', 'size'), controlled=('controlled', 'sum'), findings=('findings', 'sum'))
hit_rate = by_segment.findings / by_segment.controlled
first = hit_rate.idxmax()
```
</details>

In [ ]:
traders = pd.read_csv(DATA / 'trader_features.csv')
traders['controlled'] = traders.controlled_h1.round().astype(int)                          # lines controlled in months 1 to 18
traders['findings'] = (traders.controlled_h1 * traders.hit_rate_h1).round().astype(int)   # among them, lines where the control found something
traders[['importer_id', 'hit_rate_h1', 'mean_learned_score', 'median_share_of_median', 'log_value_12m', 'tin_blank', 'mean_weight_dev', 'importer_cluster', 'controlled', 'findings']].head()

In [ ]:
# YOUR TURN (segments)
by_segment = ...   # one row per segment: importers, controlled lines, findings
hit_rate = ...     # findings ÷ controlled lines, per segment
first = ...        # the segment a valuation or documentary team looks at first

In [ ]:
for k in (1, 2, 3, 4):
    check(f'segment {k} hit rate', None if hit_rate is ... else hit_rate[k], {"1": "297d62f39fae64ee", "2": "e7b2eca160b8791d", "3": "48052793cee5c887", "4": "385e68f77daf2e4d", "first": "4b227777d4dd1fc6"}[str(k)], 4)
check('the segment to look at first', first, '4b227777d4dd1fc6', 0)

## Recipe 2 · the description says a dearer code

The 43 lines described as "washing machines c". The **usual code** of a description is the code most lines declare. A line under a cheaper code than the usual one is the signal.

**Hint 1:** `slice2.hs8.value_counts()` counts the lines per code; `.idxmax()` gives the most common one.

<details><summary>Hint 2</summary>

```python
usual_code = slice2.hs8.value_counts().idxmax()
usual_duty = slice2[slice2.hs8 == usual_code].duty_mfn.iloc[0]
slice2['dearer_by'] = usual_duty - slice2.duty_mfn
flagged = (slice2.dearer_by >= 5).sum()
```
</details>

In [ ]:
slice2 = h1[h1.desc_clean == 'washing machines c'].copy()
slice2[['line_id', 'description', 'hs8', 'duty_mfn', 'random_draw', 'found']].head(10)

In [ ]:
# YOUR TURN (recipe 2)
usual_code = ...
usual_duty = ...
slice2['dearer_by'] = ...
flagged = ...     # lines dearer by 5 points or more

In [ ]:
check('usual code', usual_code, '5b689915d2396c15', 0)
check('lines dearer by 5 or more', flagged, '6b86b273ff34fce1', 0)

## Recipe 3 · heavier than usual per piece

Cigars and cheroots, code 2402.10.02, declared in pieces or in dozens (12 pieces to a dozen), as on the workbook's sheet. A piece much heavier than usual can hide other goods.

**Hint 1:** `np.where(slice3.unit == 'dozens', ...)` turns dozens into pieces; divide the kilos by the pieces; `.median()` gives the usual.

<details><summary>Hint 2</summary>

```python
slice3['kg_per_piece'] = slice3.net_kg / slice3.pieces
usual = slice3.kg_per_piece.median()
slice3['against_usual'] = slice3.kg_per_piece / usual
flagged = (slice3.against_usual > 1.5).sum()
```
</details>

In [ ]:
slice3 = h1[h1.hs8 == 24021002].copy()
slice3['pieces'] = np.where(slice3.unit == 'dozens', slice3.quantity * 12, slice3.quantity)   # 12 pieces to a dozen
slice3[['line_id', 'net_kg', 'quantity', 'unit', 'pieces', 'random_draw', 'found']].head()

In [ ]:
# YOUR TURN (recipe 3)
slice3['kg_per_piece'] = ...
usual = ...
slice3['against_usual'] = ...
flagged = ...     # lines more than 1.5 times the usual

In [ ]:
check('usual kilos per piece', usual, 'e4ccd7f91f75c4ba', 4)
check('lines above 1.5 times', flagged, '8527a891e2241369', 0)

## Recipe 4 · a description that says nothing

The lines described as "personal effects". Words used under many chapters do not say what the goods are; on a high duty, that is the signal.

**Hint 1:** `.nunique()` counts the different values of a column.

<details><summary>Hint 2</summary>

```python
chapters = slice4.chapter.nunique()
flagged = (slice4.duty_mfn >= 20).sum() if chapters >= 5 else 0
```
</details>

In [ ]:
slice4 = h1[h1.desc_clean == 'personal effects'].copy()
slice4[['line_id', 'description', 'chapter', 'duty_mfn', 'random_draw', 'found']].head()

In [ ]:
# YOUR TURN (recipe 4)
chapters = ...
flagged = ...     # lines on a duty of 20% or more, if the description is used under 5 chapters or more

In [ ]:
check('chapters', chapters, '2c624232cdd22177', 0)
check('lines flagged', flagged, '6f4b6612125fb3a0', 0)

## Recipe 5 · the declared value against the invoice

The random-draw lines of month 12. Put the declared value on the incoterm's basis (FOB or EXW: FOB value; CFR: FOB plus freight; CIF: customs value), turn the invoice into LCU at the month's rate, and compare.

**Hint 1:** `np.where(test, a, b)` chooses per line; `.map()` looks the rate up by currency.

<details><summary>Hint 2</summary>

```python
basis = np.where(slice5.incoterm.isin(['FOB', 'EXW']), slice5.fob_lcu,
         np.where(slice5.incoterm == 'CFR', slice5.fob_lcu + slice5.freight_lcu, slice5.customs_value_lcu))
rate = slice5.invoice_currency.map(rates)
slice5['gap'] = basis / (slice5.invoice_value * rate) - 1
flagged = (slice5.gap.abs() > 0.05).sum()
```
</details>

In [ ]:
dec = pd.read_csv(DATA / 'declarations_h1.csv', low_memory=False, usecols=['line_id', 'month', 'incoterm', 'fob_lcu', 'freight_lcu', 'customs_value_lcu', 'invoice_currency', 'invoice_value'])
slice5 = h1[(h1.period_month == 12) & (h1.random_draw == 1)][['line_id', 'found']].merge(dec, on='line_id')
fx = pd.read_csv(DATA / 'fx_rates.csv')
rates = fx[fx.month == slice5.month.iloc[0]].set_index('currency').lcu_per_unit
rates

In [ ]:
# YOUR TURN (recipe 5)
basis = ...
rate = ...
slice5['gap'] = ...
flagged = ...     # lines more than 5% away, either way

In [ ]:
check('lines more than 5% away', flagged, '4b227777d4dd1fc6', 0)

## Last section · try your setting on the whole random draw of months 1 to 18

The function below counts, for one setting, the lines flagged, the findings, the hit rate, the lift and the share of the draw (the draw is a random tenth of all lines, so its share is close to the share of traffic). Try three or four settings and keep one at or under 5%.

In [ ]:
draw1 = h1[h1.random_draw == 1].copy()
draw1['dearer_by'] = np.where(draw1.usual_code != draw1.hs8, draw1.usual_code_duty - draw1.duty_mfn, np.nan)
def try_setting(recipe, value, duty=0):
    flag = {2: draw1.dearer_by >= value, 3: draw1.kg_per_unit_ratio > value,
            4: (draw1.desc_chapters >= value) & (draw1.duty_mfn >= duty), 5: draw1.value_vs_invoice.abs() > value}[recipe]
    n, k = int(flag.sum()), int((flag & (draw1.found == 1)).sum())
    hit = k / n if n else float('nan')
    print(f'recipe {recipe}, setting {value}' + (f', duty {duty}' if recipe == 4 else '') + f': {n} lines, {k} findings, hit rate {hit:.1%}, lift {hit / draw1.found.mean():.1f}, share {n / len(draw1):.2%}')

In [ ]:
# YOUR TURN: try three or four settings of your recipe
try_setting(2, 1)
try_setting(2, 5)

Then fill the card on the site (hypothesis, rule, control and first document, share of traffic, expiry, control group, success criterion) and **lock it before any test**.